# Tutorial: One Run of the PPI / DISCount / Poisson Calibration Simulation

This notebook demonstrates **one run** of the simulation from `run_simulation_ppi_discount_poisson_calibration.py`. It loads the discount f/g data, creates a labeled/unlabeled split (using either importance sampling or random labeling), and runs all four methods:

1. **Poisson calibration** (Stan) — calibrates predicted counts to true counts
2. **Generative model** (Stan) — NegBinomial2 + detection + false positives
3. **PPI** (Prediction Powered Inference) — debiases detector predictions
4. **DISCount** — importance sampling (when q ∝ g) or sample mean (when random)

**Setup:** Use an environment with `cmdstanpy`, `numpy`, `arviz`, `ppi-python` (e.g. `conda activate stan`).

---

In [ ]:
import json
import numpy as np
import arviz as az
from cmdstanpy import CmdStanModel
from ppi_py import ppi_mean_pointestimate, ppi_mean_ci

## 1. Load the data

In [ ]:
data_path = "../data/2025-11-19_discount_f_g.json"
with open(data_path) as f:
    data_raw = json.load(f)

f_arr = np.array(data_raw["f"], dtype=np.int32)
g_arr = np.array(data_raw["g"], dtype=np.float64)
N = len(f_arr)

print(f"Loaded {N} paired observations")
print(f"f (true count):  min={f_arr.min()}, max={f_arr.max()}, mean={f_arr.mean():.2f}")
print(f"g (detector):    min={g_arr.min()}, max={g_arr.max()}, mean={g_arr.mean():.2f}")

## 2. Create labeled/unlabeled split

We use **importance sampling**: the labeled subset is sampled with probability q ∝ g (detector count). This mimics labeling effort biased toward higher-count images.

In [ ]:
n_labeled = 20
seed = 42
labeling_strategy = "importance_sampling"  # or "random"

rng = np.random.default_rng(seed)
g_floor_for_q = 1e-6
q = (g_arr + g_floor_for_q) / (g_arr + g_floor_for_q).sum()

if labeling_strategy == "importance_sampling":
    idx_labeled = rng.choice(N, size=n_labeled, replace=False, p=q)
else:
    idx_labeled = rng.choice(N, size=n_labeled, replace=False)

idx_unlabeled = np.setdiff1d(np.arange(N), idx_labeled)
n_unlabeled = N - n_labeled

f_labeled = f_arr[idx_labeled]
g_labeled = g_arr[idx_labeled]
f_unlabeled = f_arr[idx_unlabeled]
g_unlabeled = g_arr[idx_unlabeled]

print(f"Labeled:   {n_labeled} obs (f known)")
print(f"Unlabeled: {n_unlabeled} obs (f treated as unknown)")
print(f"Labeling strategy: {labeling_strategy}")

## 3. Poisson calibration (Stan)

In [ ]:
epsilon = 1e-6

data_stan = {
    "N_labeled": n_labeled,
    "N_unlabeled": n_unlabeled,
    "predicted_counts_labeled": g_labeled,
    "true_counts_labeled": f_labeled,
    "predicted_counts_unlabeled": g_unlabeled,
    "epsilon": epsilon,
}

model_poisson = CmdStanModel(stan_file="../stan_models/poisson_count_calibration.stan")
fit_poisson = model_poisson.sample(
    data=data_stan,
    seed=int(rng.integers(1, 2**31)),
    iter_sampling=500,
    iter_warmup=300,
    show_progress=True,
)

mean_rays = fit_poisson.stan_variable("mean_rays_per_image")
point_poisson = float(mean_rays.mean())
ci_poisson_lo, ci_poisson_hi = map(float, az.hdi(mean_rays, hdi_prob=0.9))

print(f"Poisson calibration: point estimate = {point_poisson:.4f}")
print(f"  90% HDI: [{ci_poisson_lo:.4f}, {ci_poisson_hi:.4f}]")

## 4. Generative model (Stan)

In [ ]:
data_generative = {
    "N_labeled": n_labeled,
    "N_unlabeled": n_unlabeled,
    "predicted_counts_labeled": g_labeled.astype(np.int32),
    "true_counts_labeled": f_labeled.astype(np.int32),
    "predicted_counts_unlabeled": g_unlabeled.astype(np.int32),
    "epsilon": epsilon,
    "use_unlabeled_likelihood": 1,
    "max_unlabeled_in_likelihood": 0,
}

model_generative = CmdStanModel(stan_file="../stan_models/generative_model.stan")
fit_gen = model_generative.sample(
    data=data_generative,
    seed=int(rng.integers(1, 2**31)),
    iter_sampling=800,
    iter_warmup=800,
    show_progress=True,
)

mean_true_count = fit_gen.stan_variable("mean_true_count")
point_generative = float(mean_true_count.mean())
ci_generative_lo, ci_generative_hi = map(float, az.hdi(mean_true_count, hdi_prob=0.9))

print(f"Generative model: point estimate = {point_generative:.4f}")
print(f"  90% HDI: [{ci_generative_lo:.4f}, {ci_generative_hi:.4f}]")

## 5. PPI (Prediction Powered Inference)

In [ ]:
if labeling_strategy == "importance_sampling":
    q_labeled = q[idx_labeled]
    w = 1.0 / np.clip(q_labeled, 1e-10, None)
    q_unlabeled = q[idx_unlabeled]
    w_unlabeled = 1.0 / np.clip(1.0 - q_unlabeled, 1e-10, None)
else:
    w = None
    w_unlabeled = None

mean_ppi = ppi_mean_pointestimate(
    Y=f_labeled.astype(np.float64),
    Yhat=g_labeled,
    Yhat_unlabeled=g_unlabeled,
    w=w,
    w_unlabeled=w_unlabeled,
)
ci_ppi = ppi_mean_ci(
    Y=f_labeled.astype(np.float64),
    Yhat=g_labeled,
    Yhat_unlabeled=g_unlabeled,
    w=w,
    w_unlabeled=w_unlabeled,
    alpha=0.1,
)

point_ppi = float(np.atleast_1d(mean_ppi)[0])
ci_ppi_lo = float(np.atleast_1d(ci_ppi[0])[0])
ci_ppi_hi = float(np.atleast_1d(ci_ppi[1])[0])

print(f"PPI: point estimate = {point_ppi:.4f}")
print(f"  90% CI: [{ci_ppi_lo:.4f}, {ci_ppi_hi:.4f}]")

## 6. DISCount (importance sampling or sample mean)

In [ ]:
n_boot = 1000

if labeling_strategy == "importance_sampling":
    q_labeled = q[idx_labeled]
    F_hat_discount = (1 / n_labeled) * (f_labeled / q_labeled).sum()
    mean_discount = F_hat_discount / N
    boot_means = []
    for _ in range(n_boot):
        idx_b = rng.integers(0, n_labeled, size=n_labeled)
        F_b = (1 / n_labeled) * (f_labeled[idx_b] / q_labeled[idx_b]).sum()
        boot_means.append(F_b / N)
else:
    mean_discount = (1 / n_labeled) * f_labeled.sum()
    boot_means = []
    for _ in range(n_boot):
        idx_b = rng.integers(0, n_labeled, size=n_labeled)
        boot_means.append((1 / n_labeled) * f_labeled[idx_b].sum())

point_discount = float(mean_discount)
ci_discount_lo, ci_discount_hi = map(float, az.hdi(np.array(boot_means), hdi_prob=0.9))

print(f"DISCount: point estimate = {point_discount:.4f}")
print(f"  90% HDI (bootstrap): [{ci_discount_lo:.4f}, {ci_discount_hi:.4f}]")

## 7. Summary: compare all methods

In [ ]:
ground_truth = f_arr.mean()

print("Mean true count E[f] — one run comparison")
print("=" * 55)
print(f"Ground truth (full data):     {ground_truth:.4f}")
print(f"Poisson calibration:         {point_poisson:.4f}   [90% CI: {ci_poisson_lo:.4f}, {ci_poisson_hi:.4f}]")
print(f"Generative model:            {point_generative:.4f}   [90% CI: {ci_generative_lo:.4f}, {ci_generative_hi:.4f}]")
print(f"PPI:                         {point_ppi:.4f}   [90% CI: {ci_ppi_lo:.4f}, {ci_ppi_hi:.4f}]")
print(f"DISCount:                    {point_discount:.4f}   [90% CI: {ci_discount_lo:.4f}, {ci_discount_hi:.4f}]")
print()
print(f"(n_labeled={n_labeled}, labeling_strategy={labeling_strategy})")